# ChromaDB Basics

**ChromaDB** is an open-source vector database built for storing and searching
*embeddings* — numeric representations of text. It lets you ask questions in
natural language and find the most semantically similar documents.

In this notebook you will learn how to:

1. Create a persistent Chroma client and a collection.
2. Add documents (Chroma embeds them automatically).
3. Run similarity searches.
4. Filter results using metadata.
5. Update, retrieve, and delete documents.
6. Manage collections (rename, list, delete) and reset the database.

> **What is an embedding?** An embedding maps text to a vector of numbers such
> that texts with similar *meaning* end up close together in vector space.
> Searching then becomes a nearest-neighbour problem.

## 1. Imports

We need the `chromadb` client, `os`/`tempfile` to build a throwaway storage
path, and `Settings` to allow resetting the database during the demo.

In [ ]:
import chromadb
from chromadb.config import Settings
import os
import tempfile

## 2. Create a client

Chroma has two common client types:

- **`PersistentClient`** – stores data on disk (used here).
- **`HttpClient`** – talks to a Chroma server over HTTP (commented out below).

We point the persistent client at a folder inside the system temp directory so
the demo is self-contained and can be safely wiped.

In [ ]:
collection_name = "Students"

client = chromadb.PersistentClient(
    path=os.path.join(tempfile.gettempdir(), "chroma_db"),
    settings=Settings(allow_reset=True),
)

# Alternatively, connect to a running Chroma server:
# client = chromadb.HttpClient(
#     host="localhost",
#     port=8000,
#     settings=Settings(allow_reset=True),
# )

## 3. Create or get a collection

A **collection** is the container that holds documents, their metadata, and
their embeddings — roughly analogous to a table in a relational database.

`get_or_create_collection` is idempotent: it creates the collection the first
time and simply returns it on later runs.

See the docs: https://docs.trychroma.com/docs/collections/configure#python

In [ ]:
collection = client.get_or_create_collection(name=collection_name)

## 4. Prepare some documents

We will work with three short, unrelated pieces of text: a student bio, a club
description, and a university description. Having distinct topics makes it easy
to see how semantic search and metadata filtering behave.

In [ ]:
student_info = """
Alexandra Thompson, a 19-year-old computer science sophomore with a 3.7 GPA,
is a member of the programming and chess clubs who enjoys pizza, swimming, and hiking
in her free time in hopes of working at a tech company after graduating from the University of Washington.
"""

club_info = """
The university chess club provides an outlet for students to come together and enjoy playing
the classic strategy game of chess. Members of all skill levels are welcome, from beginners learning
the rules to experienced tournament players. The club typically meets a few times per week to play casual games,
participate in tournaments, analyze famous chess matches, and improve members' skills.
"""

university_info = """
The University of Washington, founded in 1861 in Seattle, is a public research university
with over 45,000 students across three campuses in Seattle, Tacoma, and Bothell.
As the flagship institution of the six public universities in Washington state,
UW encompasses over 500 buildings and 20 million square feet of space,
including one of the largest library systems in the world.
"""

## 5. Add documents to the collection

`add()` requires three parallel lists:

- `documents` – the raw text to be embedded and stored.
- `metadatas` – key/value pairs used for filtering later.
- `ids` – a unique string ID for every document.

> **Default embedding function:** Chroma uses the Sentence Transformers
> `all-MiniLM-L6-v2` model to embed text. It runs **locally** on your machine
> and downloads the model files automatically the first time.
>
> Docs: https://docs.trychroma.com/docs/embeddings/embedding-functions

In [ ]:
collection.add(
    documents=[student_info, club_info, university_info],
    metadatas=[
        {"source": "student info"},
        {"source": "club info"},
        {"source": "university info"},
    ],
    ids=["id1", "id2", "id3"],
)

## 6. Similarity search

`query()` converts your natural-language question into an embedding and returns
the documents whose embeddings are closest to it. Here we ask for just the top
result.

Notice the returned payload contains `ids`, `documents`, `metadatas`, and
`distances` (a smaller distance means a closer / more similar match).

In [ ]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=1,
)
print("Query 1: ", results)

## 7. Filtering with metadata

Similarity search alone can return results from any topic. Use the `where`
parameter to constrain the search to documents matching metadata conditions.

Below we ask for 2 results but restrict them to documents whose
`source` is `"student info"`.

In [ ]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    where={"source": "student info"},  # only return documents with this metadata
)
print("Query 2: ", results)

### Combining filters with `$or` / `$and`

Chroma supports logical operators for more expressive filters. Here we search
for `"university"` but accept documents from **either** the student or the
university source.

In [ ]:
results = collection.query(
    query_texts=["university"],
    n_results=5,
    where={
        "$or": [
            {"source": "student info"},
            {"source": "university info"},
        ]
    },
)
print("Query 3: ", results)

## 8. Including embeddings and distances

By default `query()` does not return the raw vectors. Pass `include` to request
`embeddings`, `documents`, and/or `distances` explicitly — useful when you want
to inspect or reuse the vectors yourself.

In [ ]:
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    include=["embeddings", "documents", "distances"],
)
print("Query 4: ", results)

## 9. Updating a document

`update()` replaces the document text for a given ID. Chroma re-embeds the new
text automatically, so subsequent searches reflect the change.

Here we swap the student's full bio for a shorter version and re-run the query.

In [ ]:
collection.update(
    ids=["id1"],
    documents=["Kristiane Carina, a 19-year-old computer science sophomore with a 3.7 GPA"],
    metadatas=[{"source": "student info"}],
)
results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
)
print("Query 5: ", results)

## 10. Updating metadata

A few important gotchas:

- Chroma **replaces the entire metadata object** for an ID — it does **not**
  merge keys. Always pass every field you want to keep.
- Metadata values must be **strings, integers, floats, or booleans**.
  Nested dictionaries or lists are not supported.

In [ ]:
collection.update(
    ids=["id1"],  # the ID of the document you want to update
    metadatas=[
        {"source": "student info", "version": 2.0, "tags": ["python", "ai", "database"]}
    ],  # new metadata (replaces the old object)
)

## 11. Retrieving by ID and filtering

`get()` retrieves documents **without** doing a similarity search. Use it when
you already know what you want — by ID, or via metadata filters.

- `get(ids=[...])` fetches specific documents.
- `get(where=...)` filters by metadata.
- The `$contains` operator checks membership inside a list-valued metadata field.

In [ ]:
# Get by ID
results = collection.get(ids=["id1"])
print("Get by ID: ", results)

# Filter by metadata
results = collection.get(where={"source": "student info"})
print("Filter by metadata 1: ", results)

# Filter by tag (membership test inside a list)
results = collection.get(where={"tags": {"$contains": "python"}})
print("Filter by metadata 2: ", results)

## 12. Inspecting embeddings

Embeddings are just lists of floats. Let's look at the vectors returned by a
query, and then fetch the vectors for **all** documents in the collection.

In [ ]:
query_results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
    include=["embeddings", "documents", "distances"],  # request embeddings in query output
)
print("View embeddings: ", query_results)

In [ ]:
# Fetch all documents and their vector embeddings
all_data = collection.get(include=["embeddings", "documents"])

all_vectors = all_data["embeddings"]
print("All vectors: ", all_vectors)

## 13. Deleting a document

`delete(ids=[...])` removes documents from the collection. After deleting `id1`,
re-running the same query shows that the student document is gone.

In [ ]:
collection.delete(ids=["id1"])

results = collection.query(
    query_texts=["What is the student name?"],
    n_results=2,
)
print("Query after delete: ", results)

## 14. Counting and listing everything

`count()` returns the number of documents, and `get()` with no arguments
returns them all.

In [ ]:
print("Count of docs: ", collection.count())
print("All docs: ", collection.get())

## 15. Renaming a collection

`modify(name=...)` renames a collection in place. We first make sure no stale
collection with the target name exists, then rename and list the collections.

In [ ]:
if "chroma_info" in [r.name for r in client.list_collections()]:
    client.delete_collection("chroma_info")

collection.modify(name="chroma_info")

# List all collections
print("List collections: ", client.list_collections())

## 16. Deleting a collection

Deleting a collection removes it and all of its documents/embeddings.

In [ ]:
client.delete_collection(name="chroma_info")
print("List collections (after deletion): ", client.list_collections())

## 17. Resetting the database

`reset()` wipes **all** collections and data. It is handy for demos and tests
but only works with clients that allow it (not a remote `HttpClient`).

> In production, think twice before calling `reset()`!

In [ ]:
client.reset()
print("Collections after reset: ", client.list_collections())

## Summary

You have now seen the core ChromaDB workflow:

| Operation | Method |
| --- | --- |
| Create / get a collection | `client.get_or_create_collection()` |
| Add documents | `collection.add()` |
| Similarity search | `collection.query()` |
| Metadata filtering | `where={...}`, `$or` / `$and`, `$contains` |
| Retrieve without search | `collection.get()` |
| Update | `collection.update()` |
| Delete documents | `collection.delete()` |
| Manage collections | `modify()`, `list_collections()`, `delete_collection()` |
| Reset everything | `client.reset()` |

Next steps: try embedding with a custom embedding function, experiment with
different `n_results`, or load a larger corpus and compare distances.